# Automated essay scoring: feature models and rubric-guided LLMs

Independent Part 1 research project using the ELLIPSE corpus. Results and limitations are summarized in the repository README. This notebook preserves the exploratory workflow; use scripts/run_feature_models.py for a clean no-API reproduction.


## Data and feature-model evaluation

Run these cells in order. The corpus is fetched from the authors and verified by SHA-256. No API key is needed through the feature-model section.


In [ ]:
# Import all the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import json
import time
import re
from getpass import getpass

print('All libraries imported!')

In [ ]:
# Load the public ELLIPSE files from the corpus authors' repository.
# The corpus and rubric are used from upstream; they are not redistributed here.
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
import hashlib

CORPUS_BASE = 'https://raw.githubusercontent.com/scrosseye/ELLIPSE-Corpus/main/'
TRAIN_URL = CORPUS_BASE + 'ELLIPSE_Final_github_train.csv'
TEST_ZIP_URL = CORPUS_BASE + 'ELLIPSE_Final_github_test.zip'

def fetch_verified(url, expected_sha256):
    with urlopen(url, timeout=60) as response:
        data = response.read()
    actual = hashlib.sha256(data).hexdigest()
    if actual != expected_sha256:
        raise ValueError(f'Upstream file changed; review {url} before updating its SHA-256')
    return data

train_bytes = fetch_verified(TRAIN_URL, '782344e99668a3ff508d7410c0eb6e36da70f3b28f81c96e367f1ca04924b06c')
test_zip_bytes = fetch_verified(TEST_ZIP_URL, '9ecddfcda83f6a99c8a24dda47fb1c30673fedeaf578a0fadac6645fd151a2f8')
with ZipFile(BytesIO(test_zip_bytes)) as archive:
    test_bytes = archive.read('ELLIPSE_Final_github_test.csv', pwd=b'ellipse_test')

train_df = pd.read_csv(BytesIO(train_bytes))
test_df = pd.read_csv(BytesIO(test_bytes))
TRAIN_SOURCE = TRAIN_URL
TEST_SOURCE = TEST_ZIP_URL + '#ELLIPSE_Final_github_test.csv'

ID_CANDIDATES = ['essay_id', 'text_id_kaggle', 'text_id', 'id']
ESSAY_ID_COL = next((c for c in ID_CANDIDATES if c in train_df.columns and c in test_df.columns), None)

if ESSAY_ID_COL is None:
    ESSAY_ID_COL = 'essay_id'
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[ESSAY_ID_COL] = [f'train_{i:05d}' for i in range(len(train_df))]
    test_df[ESSAY_ID_COL] = [f'test_{i:05d}' for i in range(len(test_df))]

df = train_df.copy()
print(f'Training data shape: {train_df.shape}')
print(f'Official test data shape: {test_df.shape}')
print(f'Essay ID column: {ESSAY_ID_COL}')


In [ ]:
# How big are the official train/test files?
print('Training data (used for exploration below):')
print(f'  Number of essays (rows): {train_df.shape[0]}')
print(f'  Number of columns:       {train_df.shape[1]}')
print('\nOfficial test data:')
print(f'  Number of essays (rows): {test_df.shape[0]}')
print(f'  Number of columns:       {test_df.shape[1]}')

In [ ]:
# Identify which column contains the essay text
text_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Text/string columns found:', text_cols)

# Auto-detect the text column
if 'full_text' in df.columns:
    TEXT_COL = 'full_text'
elif 'text' in df.columns:
    TEXT_COL = 'text'
elif len(text_cols) > 0:
    # Use the column with the longest average string length
    avg_lens = {c: df[c].astype(str).str.len().mean() for c in text_cols}
    TEXT_COL = max(avg_lens, key=avg_lens.get)
else:
    TEXT_COL = None

print(f'\n✅ Using "{TEXT_COL}" as the essay text column.')
if TEXT_COL is None:
    print('⚠️ No text column found! LLM scoring (Section 4) will not work.')
    print('   Please check the CSV columns above and set TEXT_COL manually.')

In [ ]:
# Formal schema audit: compare columns and dtypes across official train/test
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
common_cols = sorted(train_cols & test_cols)
train_only_cols = sorted(train_cols - test_cols)
test_only_cols = sorted(test_cols - train_cols)

print('Schema audit summary')
print('-' * 60)
print(f'Common columns:      {len(common_cols)}')
print(f'Train-only columns:  {len(train_only_cols)} -> {train_only_cols if train_only_cols else "None"}')
print(f'Test-only columns:   {len(test_only_cols)} -> {test_only_cols if test_only_cols else "None"}')
print()

dtype_comparison = pd.DataFrame({
    'train_dtype': train_df[common_cols].dtypes.astype(str),
    'test_dtype': test_df[common_cols].dtypes.astype(str),
})
dtype_comparison['dtype_match'] = dtype_comparison['train_dtype'] == dtype_comparison['test_dtype']

print(f'Dtype mismatches across shared columns: {(~dtype_comparison["dtype_match"]).sum()}')
dtype_comparison


In [ ]:
# Missing-value audit for key columns and candidate AES features
score_cols = ['Overall', 'Cohesion', 'Syntax', 'Vocabulary', 'Phraseology', 'Grammar', 'Conventions']
candidate_feature_cols = [
    'num_words', 'num_sent', 'num_para', 'num_word_div_para',
    'MTLD', 'TTR', 'Type', 'Token'
]

audit_cols = []
for col in [ESSAY_ID_COL, TEXT_COL] + score_cols + candidate_feature_cols:
    if col is not None and col in train_df.columns and col in test_df.columns and col not in audit_cols:
        audit_cols.append(col)

missing_summary = pd.DataFrame({
    'train_missing': train_df[audit_cols].isna().sum(),
    'train_missing_pct': (train_df[audit_cols].isna().mean() * 100).round(2),
    'test_missing': test_df[audit_cols].isna().sum(),
    'test_missing_pct': (test_df[audit_cols].isna().mean() * 100).round(2),
}).sort_values(['train_missing', 'test_missing'], ascending=False)

print('Missing-value audit for key columns')
print('-' * 60)
print(f'Columns checked: {len(audit_cols)}')
print(f'Text column:     {TEXT_COL}')
print(f'Essay ID column: {ESSAY_ID_COL}')
print()
missing_summary


In [ ]:
# Compare score and text-length distributions across official train/test
train_text_len = train_df[TEXT_COL].astype(str).str.len()
test_text_len = test_df[TEXT_COL].astype(str).str.len()

split_summary = pd.DataFrame([
    {
        'split': 'official_train',
        'n_essays': len(train_df),
        'overall_mean': train_df['Overall'].mean(),
        'overall_std': train_df['Overall'].std(),
        'overall_min': train_df['Overall'].min(),
        'overall_max': train_df['Overall'].max(),
        'text_char_mean': train_text_len.mean(),
        'text_char_median': train_text_len.median(),
    },
    {
        'split': 'official_test',
        'n_essays': len(test_df),
        'overall_mean': test_df['Overall'].mean(),
        'overall_std': test_df['Overall'].std(),
        'overall_min': test_df['Overall'].min(),
        'overall_max': test_df['Overall'].max(),
        'text_char_mean': test_text_len.mean(),
        'text_char_median': test_text_len.median(),
    },
]).round(2)

print('Official split comparison')
print('-' * 60)
split_summary

plt.figure(figsize=(9, 4))
plt.hist(train_df['Overall'], bins=np.arange(0.75, 5.76, 0.25), alpha=0.55, label='Official train', color='steelblue', edgecolor='black')
plt.hist(test_df['Overall'], bins=np.arange(0.75, 5.76, 0.25), alpha=0.55, label='Official test', color='darkorange', edgecolor='black')
plt.xlabel('Overall Holistic Score', fontsize=12)
plt.ylabel('Number of Essays', fontsize=12)
plt.title('Overall Score Distribution: Official Train vs Official Test', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# === Evaluation function (used throughout the notebook) ===

def evaluate(y_true, y_pred, label='Model'):
    '''
    Compare predicted scores to true (human) scores.
    Returns Pearson r and Mean Squared Error (MSE).
    '''
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    # Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)

    # Pearson correlation
    # Handle edge case: if all predictions are the same number,
    # correlation is undefined (no variance), so we set r = 0.
    if np.std(y_pred) < 1e-10:
        r = 0.0
        note = ' (constant prediction → r undefined, set to 0)'
    else:
        r, _ = pearsonr(y_true, y_pred)
        note = ''

    print(f'[{label}]')
    print(f'  Pearson r = {r:.4f}{note}')
    print(f'  MSE       = {mse:.4f}')
    print()
    return r, mse

print('evaluate() function defined ✅')

In [ ]:
# Official evaluation settings
RANDOM_SEED = 42
FINAL_LLM_TEST_SIZE = 60

print(f'Official training set: {len(train_df)} essays')
print(f'Official test set:     {len(test_df)} essays')
print()
print(f'Final LLM evaluation subset size (official test): {FINAL_LLM_TEST_SIZE} essays')


### Baselines on the full official test split

Constant, random, and length-based scores provide context for the two feature regressions. Random scores use a fixed seed.


In [ ]:
# Ground truth: the human Overall scores for the official test set
y_true_official = test_df['Overall'].values

# -------------------------------------------------------
# Baseline A1: Always predict the LOWEST score (1.0)
# -------------------------------------------------------
y_pred_a1 = np.ones(len(y_true_official)) * 1.0
r_a1, mse_a1 = evaluate(y_true_official, y_pred_a1, 'Baseline A1: Always 1.0')

# -------------------------------------------------------
# Baseline A2: Always predict the HIGHEST score (5.0)
# -------------------------------------------------------
y_pred_a2 = np.ones(len(y_true_official)) * 5.0
r_a2, mse_a2 = evaluate(y_true_official, y_pred_a2, 'Baseline A2: Always 5.0')

# -------------------------------------------------------
# Baseline A3: Always predict the MEAN score from training
# -------------------------------------------------------
train_mean = train_df['Overall'].mean()
y_pred_a3 = np.ones(len(y_true_official)) * train_mean
r_a3, mse_a3 = evaluate(y_true_official, y_pred_a3, f'Baseline A3: Always {train_mean:.2f} (train mean)')


In [ ]:
# -------------------------------------------------------
# Baseline B: Random scores between 1.0 and 5.0
# -------------------------------------------------------
np.random.seed(123)
y_pred_b = np.random.uniform(1.0, 5.0, size=len(y_true_official))
r_b, mse_b = evaluate(y_true_official, y_pred_b, 'Baseline B: Random (uniform 1-5)')


In [ ]:
# -------------------------------------------------------
# Baseline C: Use raw word count as the "score"
# -------------------------------------------------------
# This is clearly wrong (word counts are in the hundreds,
# but scores are 1-5), but it shows what happens when
# your predictions are on the wrong scale.
y_pred_c = test_df['num_words'].values.astype(float)
r_c, mse_c = evaluate(y_true_official, y_pred_c, 'Baseline C: Raw word count (wrong scale!)')

print('Notice: Pearson r might be OK (word count correlates with proficiency),')
print('but MSE is huge because the scale is completely wrong!')


In [ ]:
# -------------------------------------------------------
# Baseline D: Normalize word count to the 1-5 range
# -------------------------------------------------------
# Min-max normalization: rescale values from [min, max] to [1, 5]
# Formula: new = 1 + 4 * (old - min) / (max - min)

wc_min = train_df['num_words'].min()
wc_max = train_df['num_words'].max()

y_pred_d = 1.0 + 4.0 * (test_df['num_words'].values - wc_min) / (wc_max - wc_min)
y_pred_d = np.clip(y_pred_d, 1.0, 5.0)

r_d, mse_d = evaluate(y_true_official, y_pred_d, 'Baseline D: Normalized word count (1-5)')

print('Now the scale is correct, so MSE is much better.')
print('But can we do better with more features? -> Section 3!')


In [ ]:
# -------------------------------------------------------
# Summary table of all baselines
# -------------------------------------------------------
print('=' * 60)
print(f'{"Baseline":<42} {"Pearson r":>9} {"MSE":>8}')
print('=' * 60)
baselines = [
    ('A1: Always predict 1.0', r_a1, mse_a1),
    ('A2: Always predict 5.0', r_a2, mse_a2),
    (f'A3: Always predict {train_mean:.2f} (mean)', r_a3, mse_a3),
    ('B: Random (uniform 1-5)', r_b, mse_b),
    ('C: Raw word count (wrong scale)', r_c, mse_c),
    ('D: Normalized word count (1-5)', r_d, mse_d),
]
for name, r, mse in baselines:
    print(f'  {name:<40} {r:>9.4f} {mse:>8.4f}')
print('=' * 60)

In [ ]:
# Build LR-basic and LR-expanded feature matrices

def add_engineered_features(frame):
    frame = frame.copy()
    frame['avg_sentence_length'] = frame['num_words'] / frame['num_sent'].clip(lower=1)
    frame['avg_paragraph_length'] = frame['num_words'] / frame['num_para'].clip(lower=1)
    frame['sentences_per_paragraph'] = frame['num_sent'] / frame['num_para'].clip(lower=1)
    frame['log_num_words'] = np.log1p(frame['num_words'])
    return frame

train_lr_df = add_engineered_features(train_df)
test_lr_df = add_engineered_features(test_df)

LR_BASIC_FEATURES = ['num_words', 'num_sent', 'TTR', 'MTLD', 'num_word_div_para']
LR_EXPANDED_FEATURES = LR_BASIC_FEATURES + [
    'num_para',
    'avg_sentence_length',
    'avg_paragraph_length',
    'sentences_per_paragraph',
    'log_num_words',
    'Type',
    'Token',
]

all_lr_features = LR_BASIC_FEATURES + [f for f in LR_EXPANDED_FEATURES if f not in LR_BASIC_FEATURES]
missing_feature_cols = [
    f for f in all_lr_features
    if f not in train_lr_df.columns or f not in test_lr_df.columns
]
if missing_feature_cols:
    raise ValueError(f'Missing feature columns for regression: {missing_feature_cols}')

X_train_basic = train_lr_df[LR_BASIC_FEATURES].values
y_train = train_lr_df['Overall'].values
X_test_basic = test_lr_df[LR_BASIC_FEATURES].values
y_test = test_lr_df['Overall'].values

X_train_expanded = train_lr_df[LR_EXPANDED_FEATURES].values
X_test_expanded = test_lr_df[LR_EXPANDED_FEATURES].values

print('LR-basic features:', LR_BASIC_FEATURES)
print('LR-expanded features:', LR_EXPANDED_FEATURES)
print()
print(f'Official train size: {len(train_lr_df)} essays')
print(f'Official test size:  {len(test_lr_df)} essays')
print(f'LR-basic matrix:     train {X_train_basic.shape}, test {X_test_basic.shape}')
print(f'LR-expanded matrix:  train {X_train_expanded.shape}, test {X_test_expanded.shape}')


In [ ]:
# Train both linear regression variants
lr_basic_model = LinearRegression()
lr_basic_model.fit(X_train_basic, y_train)

lr_expanded_model = LinearRegression()
lr_expanded_model.fit(X_train_expanded, y_train)

print('LR-basic model trained successfully.')
print('LR-expanded model trained successfully.')


In [ ]:
# Predict on the official test split and evaluate both models
y_pred_lr_basic = np.clip(lr_basic_model.predict(X_test_basic), 1.0, 5.0)
y_pred_lr_expanded = np.clip(lr_expanded_model.predict(X_test_expanded), 1.0, 5.0)

r_lr_basic, mse_lr_basic = evaluate(y_test, y_pred_lr_basic, 'LR-basic (5 features)')
r_lr_expanded, mse_lr_expanded = evaluate(y_test, y_pred_lr_expanded, 'LR-expanded (12 features)')

lr_candidates = [
    {
        'name': 'LR-basic',
        'model': lr_basic_model,
        'features': LR_BASIC_FEATURES,
        'predictions': y_pred_lr_basic,
        'pearson_r': r_lr_basic,
        'mse': mse_lr_basic,
    },
    {
        'name': 'LR-expanded',
        'model': lr_expanded_model,
        'features': LR_EXPANDED_FEATURES,
        'predictions': y_pred_lr_expanded,
        'pearson_r': r_lr_expanded,
        'mse': mse_lr_expanded,
    },
]

selected_lr = max(lr_candidates, key=lambda item: (item['pearson_r'], -item['mse']))
selected_lr_name = selected_lr['name']
lr_model = selected_lr['model']
FEATURES = selected_lr['features']
y_pred_lr = selected_lr['predictions']
r_lr = selected_lr['pearson_r']
mse_lr = selected_lr['mse']

print(f'Selected default export model: {selected_lr_name}')
print(f'Selected feature count:        {len(FEATURES)}')


In [ ]:
# Inspect feature weights for both linear regression variants

def print_coefficient_table(label, feature_names, model):
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Weight': model.coef_,
    })
    coef_df['AbsWeight'] = coef_df['Weight'].abs()
    coef_df = coef_df.sort_values('AbsWeight', ascending=False).drop(columns='AbsWeight')

    print(label)
    print('-' * 60)
    print(coef_df.to_string(index=False))
    print(f'Intercept: {model.intercept_:.6f}')
    print()

print_coefficient_table('LR-basic coefficients', LR_BASIC_FEATURES, lr_basic_model)
print_coefficient_table('LR-expanded coefficients', LR_EXPANDED_FEATURES, lr_expanded_model)
print(f'Default export model for later cells: {selected_lr_name}')


In [ ]:
# Scatter plots: LR-basic vs LR-expanded on the official test split
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
plot_specs = [
    ('LR-basic', y_pred_lr_basic, r_lr_basic, 'steelblue'),
    ('LR-expanded', y_pred_lr_expanded, r_lr_expanded, 'darkorange'),
]

for ax, (label, preds, corr, color) in zip(axes, plot_specs):
    ax.scatter(y_test, preds, alpha=0.45, edgecolors='black', linewidths=0.4, color=color)
    ax.plot([1, 5], [1, 5], 'r--', linewidth=1)
    ax.set_title(f'{label} (r = {corr:.3f})', fontsize=12)
    ax.set_xlabel('Human Score')
    ax.set_ylabel('Predicted Score')
    ax.set_xlim(0.5, 5.5)
    ax.set_ylim(0.5, 5.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Linear Regression Comparison on the Official Test Split', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Updated comparison table: baselines + both linear regression variants
print('=' * 68)
print(f'{"Model":<46} {"Pearson r":>9} {"MSE":>8}')
print('=' * 68)
all_results = baselines + [
    ('LR-basic (5 features)', r_lr_basic, mse_lr_basic),
    ('LR-expanded (12 features)', r_lr_expanded, mse_lr_expanded),
]
for name, r, mse in all_results:
    print(f'  {name:<44} {r:>9.4f} {mse:>8.4f}')
print('=' * 68)
print()
print(f'Default export model for later cells: {selected_lr_name}')
print('Next, we can move on to the official-test LLM scoring section.')


### Optional paid LLM experiment

The rubric is fetched from upstream. API calls require an explicit opt-in in the client setup cell. The full 12-condition run sends 720 initial requests, with up to two retries per failure. Check current provider pricing and account limits first.


In [ ]:
# Paid API use requires an explicit opt-in.
import os
from openai import OpenAI

RUN_PAID_LLM = False  # Change to True only after checking price and account limits.
if not RUN_PAID_LLM:
    raise RuntimeError('Paid LLM experiment is disabled. Set RUN_PAID_LLM=True in this cell to continue.')
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise RuntimeError('Set OPENROUTER_API_KEY in the environment before starting the paid experiment.')
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=OPENROUTER_API_KEY)
print('OpenRouter client ready.')


In [ ]:
# Read the official scoring rubric from the corpus authors' repository.
# This avoids bundling a third-party document into this portfolio repository.
from zipfile import ZipFile
from xml.etree import ElementTree as ET

ALLOWED_SCORES = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
RUBRIC_URL = CORPUS_BASE + 'ELL_Rubrics.docx'
rubric_bytes = fetch_verified(RUBRIC_URL, '0c0ac5dfa6ae89c9c99ffe483c13ea4a57e8152a1472797e1fe361c2c203efdc')

def extract_docx_table_rows(docx_bytes):
    with ZipFile(BytesIO(docx_bytes)) as archive:
        xml_bytes = archive.read('word/document.xml')
    root = ET.fromstring(xml_bytes)
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}

    def cell_text(tc):
        texts = [t.text for t in tc.findall('.//w:t', ns) if t.text]
        return ' '.join(''.join(texts).split())

    tables = []
    for tbl in root.findall('.//w:tbl', ns):
        rows = []
        for tr in tbl.findall('./w:tr', ns):
            cells = [cell_text(tc) for tc in tr.findall('./w:tc', ns)]
            if any(cells):
                rows.append(cells)
        if rows:
            tables.append(rows)
    return tables

def build_holistic_rubric_from_docx(docx_bytes):
    tables = extract_docx_table_rows(docx_bytes)
    if not tables or len(tables[0]) < 3:
        raise ValueError('The upstream rubric has no expected table structure')
    rubric_table = tables[0]
    header_row = rubric_table[1]
    if 'Overall' not in header_row:
        raise ValueError('The upstream rubric has no Overall column')
    overall_col_idx = header_row.index('Overall')
    score_map = {}
    for row in rubric_table[2:]:
        if len(row) <= overall_col_idx:
            continue
        score = row[0].strip()
        overall_text = row[overall_col_idx].strip()
        if score in {'5', '4', '3', '2', '1'} and overall_text:
            score_map[score] = overall_text
    if set(score_map) != {'5', '4', '3', '2', '1'}:
        raise ValueError('The upstream rubric is missing a holistic score band')
    lines = ['Official holistic rubric extracted from ELL_Rubrics.docx:']
    for score in ['5', '4', '3', '2', '1']:
        lines.append(f"{score}.0 - {score_map[score]}")
    lines.append('Use 4.5, 3.5, 2.5, and 1.5 for essays that fall between adjacent score bands.')
    return '\n'.join(lines)

RUBRIC_TEXT = build_holistic_rubric_from_docx(rubric_bytes)
RUBRIC_SOURCE = RUBRIC_URL

PROMPT_LIBRARY = {
    'P1_direct_holistic': f"""You are an expert assessor of English Language Learner (ELL) writing.

Your job is to assign one holistic proficiency score to the student's essay using the rubric below.

{RUBRIC_TEXT}

Instructions:
- Score only the quality of the student's English writing.
- Focus on language proficiency, clarity, grammar, syntax, vocabulary, cohesion, and organization.
- Do not judge the essay based on whether you agree with the student's opinion.
- Do not judge the essay based on factual correctness unless it directly affects comprehensibility.
- Use only these allowed scores: {ALLOWED_SCORES}
- Respond with exactly one score and no explanation.""",

    'P2_analytic_to_holistic': f"""You are an expert assessor of English Language Learner (ELL) writing.

Assign one holistic proficiency score to the student's essay using the rubric below.

{RUBRIC_TEXT}

Before choosing the final score, internally consider:
- comprehensibility
- organization and cohesion
- grammar and syntax control
- vocabulary range and word choice
- sentence variety and fluency

Then map your internal judgment to one holistic score.

Instructions:
- Score only the quality of the student's English writing.
- Do not judge the essay based on opinion quality or topic stance.
- Use only these allowed scores: {ALLOWED_SCORES}
- Respond with exactly one score and no explanation.""",
}

print(f'Loaded official rubric from: {RUBRIC_SOURCE}')
print('Prompt versions ready:', list(PROMPT_LIBRARY))


In [ ]:
# OpenRouter model registry for the final 12-condition comparison
MODEL_REGISTRY = {
    'deepseek_v3_2': 'deepseek/deepseek-v3.2',
    'mistral_small_3_1_24b': 'mistralai/mistral-small-3.1-24b-instruct',
    'gemma_3_27b_it': 'google/gemma-3-27b-it',
}

print('Registered OpenRouter models:')
for short_name, model_id in MODEL_REGISTRY.items():
    print(f'  {short_name:<24} -> {model_id}')


In [ ]:
# Experiment settings: 3 models x 2 prompts x 2 temperatures = 12 conditions
TEMPERATURES = [0.0, 0.2]
MAX_TOKENS = 8
TOP_P = 1.0
REQUEST_DELAY_SECONDS = 0.4
MAX_RETRIES = 2
LLM_FINAL_SIZE = FINAL_LLM_TEST_SIZE

_, llm_eval_df = train_test_split(
    test_df,
    test_size=LLM_FINAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=test_df['Overall'],
)
llm_eval_df = llm_eval_df.reset_index(drop=True).copy()

EXPERIMENT_CONFIGS = []
for model_key, model_id in MODEL_REGISTRY.items():
    for prompt_key, prompt_text in PROMPT_LIBRARY.items():
        for temperature in TEMPERATURES:
            temp_tag = str(temperature).replace('.', 'p')
            experiment_id = f'{model_key}__{prompt_key}__t{temp_tag}'
            EXPERIMENT_CONFIGS.append({
                'experiment_id': experiment_id,
                'model_key': model_key,
                'model_id': model_id,
                'prompt_key': prompt_key,
                'system_prompt': prompt_text,
                'temperature': temperature,
            })

print(f'Official test subset size for final LLM evaluation: {len(llm_eval_df)} essays')
print('Score distribution in llm_eval_df (official test subset):')
print(llm_eval_df['Overall'].value_counts().sort_index())
print()
print(f'Total experiment conditions: {len(EXPERIMENT_CONFIGS)}')
for config in EXPERIMENT_CONFIGS:
    print(f"  {config['experiment_id']}")


In [ ]:
# Strict response parser and request helper.
def parse_llm_score(reply, allowed_scores=ALLOWED_SCORES):
    if reply is None:
        return None
    value = str(reply).strip()
    if not re.fullmatch(r'[1-5](?:\.0|\.5)?', value):
        return None
    score = float(value)
    return score if score in allowed_scores else None

def score_essay_with_llm(essay_text, client, model, system_prompt, temperature, max_tokens, top_p=1.0):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Score this essay. Return only one score.\n\nEssay:\n{essay_text}'},
            ],
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            stream=False,
        )
        reply = response.choices[0].message.content
        if not isinstance(reply, str):
            return None, 'ERROR: non-text response'
        reply = reply.strip()
        score = parse_llm_score(reply)
        return score, reply
    except Exception as exc:
        return None, f'ERROR: {type(exc).__name__}: {exc}'

print('LLM request and strict parsing helpers ready.')


In [ ]:
# Sequential runner for one experiment configuration

def run_llm_experiment(eval_df, client, config, max_tokens, top_p, request_delay_seconds, max_retries):
    scores = []
    raw_replies = []
    failed_local_indices = []

    for local_idx, row in eval_df.reset_index(drop=True).iterrows():
        essay_text = str(row[TEXT_COL])
        score = None
        raw_reply = None

        for attempt in range(max_retries + 1):
            score, raw_reply = score_essay_with_llm(
                essay_text=essay_text,
                client=client,
                model=config['model_id'],
                system_prompt=config['system_prompt'],
                temperature=config['temperature'],
                max_tokens=max_tokens,
                top_p=top_p,
            )
            if score is not None:
                break
            time.sleep(request_delay_seconds * (attempt + 1))

        if score is None:
            failed_local_indices.append(local_idx)

        scores.append(score)
        raw_replies.append(raw_reply)
        time.sleep(request_delay_seconds)

    valid_mask = np.array([score is not None for score in scores])
    y_true = eval_df['Overall'].values[valid_mask]
    y_pred = np.array([score for score in scores if score is not None], dtype=float)

    if len(y_pred) >= 2:
        r_value, mse_value = evaluate(
            y_true,
            y_pred,
            label=f"{config['model_key']} | {config['prompt_key']} | t={config['temperature']}"
        )
    else:
        r_value, mse_value = np.nan, np.nan
        print(f"[{config['experiment_id']}] Not enough valid predictions to compute metrics.\n")

    return {
        'essay_ids': eval_df[ESSAY_ID_COL].tolist(),
        'scores': scores,
        'raw_replies': raw_replies,
        'failed_indices': failed_local_indices,
        'successful_count': int(valid_mask.sum()),
        'pearson_r': None if pd.isna(r_value) else float(r_value),
        'mse': None if pd.isna(mse_value) else float(mse_value),
    }

print('run_llm_experiment() helper ready.')


In [ ]:
# Run all 12 OpenRouter LLM experiments on the official-test subset
llm_experiment_results = {}
llm_result_rows = []

for run_idx, config in enumerate(EXPERIMENT_CONFIGS, start=1):
    print('=' * 88)
    print(f"Running experiment {run_idx}/{len(EXPERIMENT_CONFIGS)}: {config['experiment_id']}")
    print(f"Model: {config['model_id']}")
    print(f"Prompt: {config['prompt_key']} | Temperature: {config['temperature']}")
    print('=' * 88)

    experiment_result = run_llm_experiment(
        eval_df=llm_eval_df,
        client=client,
        config=config,
        max_tokens=MAX_TOKENS,
        top_p=TOP_P,
        request_delay_seconds=REQUEST_DELAY_SECONDS,
        max_retries=MAX_RETRIES,
    )
    llm_experiment_results[config['experiment_id']] = {
        'config': config,
        **experiment_result,
    }

    llm_result_rows.append({
        'experiment_id': config['experiment_id'],
        'model_key': config['model_key'],
        'model_id': config['model_id'],
        'prompt_key': config['prompt_key'],
        'temperature': config['temperature'],
        'successful_count': experiment_result['successful_count'],
        'num_failed': len(experiment_result['failed_indices']),
        'pearson_r': experiment_result['pearson_r'],
        'mse': experiment_result['mse'],
    })

llm_results_df = pd.DataFrame(llm_result_rows)
llm_results_df = llm_results_df.sort_values(['pearson_r', 'mse'], ascending=[False, True], na_position='last').reset_index(drop=True)
print('\nOfficial-test LLM runs completed.')


### LLM results on the same 60 official-test essays

The selected condition is chosen by highest Pearson r, with MSE as a tie-breaker. Because selection and reporting use the same 60 essays, treat these LLM results as exploratory.


In [ ]:
# Summarize the 12 LLM conditions and select the best one for later export
summary_cols = ['experiment_id', 'model_key', 'prompt_key', 'temperature', 'successful_count', 'num_failed', 'pearson_r', 'mse']
print('LLM official-test summary (sorted by Pearson r, then MSE):')
print(llm_results_df[summary_cols].to_string(index=False))

best_llm_row = llm_results_df.iloc[0]
best_experiment_id = best_llm_row['experiment_id']
best_llm_result = llm_experiment_results[best_experiment_id]

MODEL_NAME = best_llm_result['config']['model_id']
TEMPERATURE = best_llm_result['config']['temperature']
SYSTEM_PROMPT = best_llm_result['config']['system_prompt']
PROMPT_VERSION = best_llm_result['config']['prompt_key']
llm_scores = best_llm_result['scores']
llm_raw_replies = best_llm_result['raw_replies']
failed_indices = best_llm_result['failed_indices']

y_true_llm = np.array(llm_eval_df['Overall'].values, dtype=float)
y_pred_llm = np.array(llm_scores, dtype=float)
r_llm = best_llm_result['pearson_r']
mse_llm = best_llm_result['mse']

print()
print('Selected best LLM configuration for final export:')
print(f'  Experiment ID: {best_experiment_id}')
print(f'  Model:         {MODEL_NAME}')
print(f'  Prompt:        {PROMPT_VERSION}')
print(f'  Temperature:   {TEMPERATURE}')
print(f'  Pearson r:     {r_llm:.4f}')
print(f'  MSE:           {mse_llm:.4f}')


In [ ]:
# Scatter plot: best OpenRouter LLM configuration vs. human scores
plt.figure(figsize=(6.5, 5))
plt.scatter(y_true_llm, y_pred_llm, alpha=0.55, edgecolors='black', linewidths=0.4, color='darkorange')
plt.plot([1, 5], [1, 5], 'r--', linewidth=1)
plt.xlabel('Human Score')
plt.ylabel('Predicted Score')
plt.title(f'Best OpenRouter LLM vs. Human Scores (Official Test Subset, r = {r_llm:.3f})', fontsize=13)
plt.xlim(0.5, 5.5)
plt.ylim(0.5, 5.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Keep full-test and matched-subset results in explicitly labeled rows.
comparison_rows = [
    {'method': 'LR-basic', 'scope': 'full official test', 'n': len(y_test), 'Pearson r': r_lr_basic, 'MSE': mse_lr_basic},
    {'method': 'LR-expanded', 'scope': 'full official test', 'n': len(y_test), 'Pearson r': r_lr_expanded, 'MSE': mse_lr_expanded},
]
subset_index = test_df[ESSAY_ID_COL].isin(set(llm_eval_df[ESSAY_ID_COL])).to_numpy()
subset_y = y_test[subset_index]
for name, predictions in [('LR-basic', y_pred_lr_basic), ('LR-expanded', y_pred_lr_expanded)]:
    subset_r, subset_mse = evaluate(subset_y, predictions[subset_index], f'{name}, matched 60')
    comparison_rows.append({'method': name, 'scope': 'matched 60', 'n': len(subset_y), 'Pearson r': subset_r, 'MSE': subset_mse})
comparison_rows.append({'method': f'{MODEL_NAME} ({PROMPT_VERSION}, t={TEMPERATURE})', 'scope': 'matched 60', 'n': len(y_true_llm), 'Pearson r': r_llm, 'MSE': mse_llm})
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))


In [ ]:
# Save predictions to CSV
results_df = test_df[[ESSAY_ID_COL, 'Overall']].copy()
results_df = results_df.rename(columns={ESSAY_ID_COL: 'essay_id', 'Overall': 'human_score'})
results_df['evaluation_split'] = 'official_test'
results_df['llm_final_selected'] = results_df['essay_id'].isin(set(llm_eval_df[ESSAY_ID_COL])).astype(int)
results_df['lr_basic_prediction'] = y_pred_lr_basic
results_df['lr_expanded_prediction'] = y_pred_lr_expanded
results_df['lr_prediction'] = y_pred_lr

for experiment_id, experiment_payload in llm_experiment_results.items():
    score_map = dict(zip(experiment_payload['essay_ids'], experiment_payload['scores']))
    results_df[f'{experiment_id}__score'] = results_df['essay_id'].map(score_map)

best_score_map = dict(zip(best_llm_result['essay_ids'], llm_scores))
best_reply_map = dict(zip(best_llm_result['essay_ids'], llm_raw_replies))
results_df['llm_prediction'] = results_df['essay_id'].map(best_score_map)
results_df['llm_raw_reply'] = results_df['essay_id'].map(best_reply_map)

results_df.to_csv('part1_predictions.csv', index=False)
print('Saved predictions to part1_predictions.csv')
print(f'   {len(results_df)} rows saved.')
print()
results_df.head(10)


In [ ]:
# Save model configuration to JSON
config = {
    'project': 'CALL Part 1: Automated Essay Scoring',
    'data': {
        'train_source': TRAIN_SOURCE,
        'test_source': TEST_SOURCE,
        'official_train_size': len(train_df),
        'official_test_size': len(test_df),
        'random_seed': RANDOM_SEED,
        'essay_id_column': ESSAY_ID_COL,
        'text_column': TEXT_COL,
    },
    'linear_regression': {
        'selected_variant': selected_lr_name,
        'selected_features': FEATURES,
        'selected_coefficients': lr_model.coef_.tolist(),
        'selected_intercept': float(lr_model.intercept_),
        'selected_pearson_r': round(float(r_lr), 4),
        'selected_mse': round(float(mse_lr), 4),
        'evaluation_split': 'official_test',
        'variants': {
            'lr_basic': {
                'features': LR_BASIC_FEATURES,
                'coefficients': lr_basic_model.coef_.tolist(),
                'intercept': float(lr_basic_model.intercept_),
                'pearson_r': round(float(r_lr_basic), 4),
                'mse': round(float(mse_lr_basic), 4),
            },
            'lr_expanded': {
                'features': LR_EXPANDED_FEATURES,
                'coefficients': lr_expanded_model.coef_.tolist(),
                'intercept': float(lr_expanded_model.intercept_),
                'pearson_r': round(float(r_lr_expanded), 4),
                'mse': round(float(mse_lr_expanded), 4),
            },
        },
    },
    'llm': {
        'provider': 'OpenRouter',
        'api_base_url': 'https://openrouter.ai/api/v1',
        'evaluation_split': 'official_test_subset',
        'evaluation_subset_size': len(llm_eval_df),
        'evaluation_essay_ids': llm_eval_df[ESSAY_ID_COL].tolist(),
        'registered_models': MODEL_REGISTRY,
        'prompt_versions': list(PROMPT_LIBRARY.keys()),
        'temperatures': TEMPERATURES,
        'max_tokens': MAX_TOKENS,
        'top_p': TOP_P,
        'request_delay_seconds': REQUEST_DELAY_SECONDS,
        'max_retries': MAX_RETRIES,
        'selected_experiment_id': best_experiment_id,
        'selected_model_name': MODEL_NAME,
        'selected_prompt_version': PROMPT_VERSION,
        'selected_temperature': TEMPERATURE,
        'system_prompt': SYSTEM_PROMPT,
        'pearson_r': round(float(r_llm), 4),
        'mse': round(float(mse_llm), 4),
        'num_successfully_scored': int(len(y_pred_llm)),
        'num_failed': len(failed_indices),
        'experiments': llm_results_df.to_dict(orient='records'),
    },
}

with open('part1_config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('Saved configuration to part1_config.json')
print()

# Display the config
print('Configuration saved; row-level IDs are not displayed.')
